In [17]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

pd.set_option("display.max_columns",50)
pd.set_option("display.width",200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "Customer_Transactions"

In [3]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type= 'table';")
print("Tables in database:", cursor.fetchall())

Tables in database: [('Customer_Transactions',)]


In [ ]:
df = pd.read_sql_query("SELECT * FROM Customer_Transactions",con)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Loaded 580,962 rows and 14 columns.


In [11]:
print("Shape (rows, columns):", df.shape)
print("Total rows:", f"{df.shape[0]:,}")
print("Total columns:", df.shape[1])

Shape (rows, columns): (580962, 14)
Total rows: 580,962
Total columns: 14


In [20]:
col_info = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Null %":(df.isna().mean()* 100).round(2).values,
    "Unique": df.nunique(dropna=True).values,
})
print(col_info.to_string(index=True))

                 Column    Dtype  Non-Null   Null  Null %  Unique
0         TransactionID      str    580962      0    0.00  580962
1            CustomerID      str    580962      0    0.00    4997
2       TransactionDate      str    580962      0    0.00     639
3       TransactionTime      str    580962      0    0.00   71454
4       TransactionType      str    580962      0    0.00       9
5     TransactionAmount  float64    580962      0    0.00  306886
6   TransactionCurrency      str    569116  11846    2.04       6
7             AssetType      str    566487  14475    2.49       5
8           CustomerDOB      str    580962      0    0.00    4393
9            CustGender      str    580962      0    0.00       2
10         CustLocation      str    563766  17196    2.96      23
11   CustAccountBalance  float64    580962      0    0.00  407472
12       WalletActivity    int64    580962      0    0.00       2
13    TransactionStatus      str    580962      0    0.00       5


In [19]:
df.head(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,NaN,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
5,TXN00000006,CUST00001,2024-12-15,07:10:38,Crypto Buy,74.81,USDT,USDT,2007-04-30,Male,London,"4,172.19",1,Completed
6,TXN00000007,CUST00001,2024-12-15,11:41:13,Cross-Border Payment,441.29,EUR,Traditional,2007-04-30,Male,London,"3,797.09",0,Completed
7,TXN00000008,CUST00001,2024-12-15,19:27:06,Deposit,275.91,EUR,Traditional,2007-04-30,Male,London,"4,031.62",0,Completed
8,TXN00000009,CUST00001,2024-12-16,17:44:56,Deposit,169.85,USD,Traditional,2007-04-30,Male,London,"4,164.10",0,Completed
9,TXN00000010,CUST00001,2024-12-21,08:17:59,P2P Transfer,182.63,USD,NaN,2007-04-30,Male,London,"4,021.65",0,Completed


In [21]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_cols)
df[numeric_cols].describe().T

Numeric columns: ['TransactionAmount', 'CustAccountBalance', 'WalletActivity']


,count,mean,std,min,25%,50%,75%,max
TransactionAmount,"580,962.00","4,461.42","17,257.31",0.00,100.31,366.32,"1,177.34","320,512.82"
CustAccountBalance,"580,962.00","24,059.17","71,073.50",0.00,716.88,"2,781.76","8,977.33","1,263,280.33"
WalletActivity,"580,962.00",0.41,0.49,0.00,0.00,0.00,1.00,1.00


In [23]:
missing = pd.DataFrame ({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(3) 
}).sort_values("Missing Count", ascending= False)

print("Total missing values in dataset:", df.isna().sum().sum())
missing

Total missing values in dataset: 43517


,Missing Count,Missing %
CustLocation,17196,2.96
AssetType,14475,2.49
TransactionCurrency,11846,2.04
TransactionDate,0,0.00
TransactionID,0,0.00
CustomerID,0,0.00
TransactionAmount,0,0.00
TransactionType,0,0.00
TransactionTime,0,0.00
CustomerDOB,0,0.00


In [24]:
duplicate_rows =df.duplicated().sum()
duplicate_txnids =df['TransactionID'].duplicated().sum()
duplicate_custids =df['CustomerID'].duplicated().sum()

print(f"Fully duplicated rows: {duplicate_rows}")
print(f"Duplicated TransactionIDs: {duplicate_txnids}")
print(f"Duplicated CustomerIDs (expected - one customer, many txns): {duplicate_custids:,}")

Fully duplicated rows: 0
Duplicated TransactionIDs: 0
Duplicated CustomerIDs (expected - one customer, many txns): 575,965


In [25]:
print("Unique customers:", df["CustomerID"].nunique())
print("Total Transactions:", len(df))
print("Avg transactions per customer:",
      round(len(df) / df["CustomerID"].nunique(),2))

Unique customers: 4997
Total Transactions: 580962
Avg transactions per customer: 116.26


In [26]:
print("Unique TransactionID:", df["TransactionID"].nunique())
print("Unique CustomerID:", df["CustomerID"].nunique())
print("Unique TransactionType:", df["TransactionType"].nunique())
print("Unique TransactionCurrency:", df["TransactionCurrency"].nunique())
print("Unique AssetType:", df["AssetType"].nunique())
print("Unique CustLocation:", df["CustLocation"].nunique())
print("Unique TransactionStatus:", df["TransactionStatus"].nunique())
print("Unique CustGender:", df["CustGender"].nunique())


Unique TransactionID: 580962
Unique CustomerID: 4997
Unique TransactionType: 9
Unique TransactionCurrency: 6
Unique AssetType: 5
Unique CustLocation: 23
Unique TransactionStatus: 5
Unique CustGender: 2


In [27]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])
df["CustomerDOB"] =pd.to_datetime(df["CustomerDOB"])

print("Earliest TransactionDate:", df["TransactionDate"].min())
print("Latest TransactionDate:", df["TransactionDate"].max())
print("Date span (days):", (df["TransactionDate"].max() - df["TransactionDate"].min()).days)

print("\n Earliest CustomerDOB:", df["CustomerDOB"].min())
print("Latest CustomerDOB:", df["CustomerDOB"].max())

print("\nTransactionTime sample:", df["TransactionTime"].head(3).tolist())


Earliest TransactionDate: 2024-12-13 00:00:00
Latest TransactionDate: 2026-09-12 00:00:00
Date span (days): 638

 Earliest CustomerDOB: 1955-10-08 00:00:00
Latest CustomerDOB: 2008-09-12 00:00:00

TransactionTime sample: ['10:03:23', '17:15:53', '12:26:16']


In [28]:
txns_per_cust = df.groupby("CustomerID").size()

print("Transactions per customer - summary: ")
print(txns_per_cust.describe().round(2))

Transactions per customer - summary: 
count   4,997.00
mean      116.26
std        95.47
min         1.00
25%        16.00
50%       128.00
75%       177.00
max       487.00
dtype: float64


In [ ]:
top_cust = (df.groupby("CustomerID")
            .agg(txn_count= ("TransactionID", "count"),
                 total_amount = ("TransactionAmount","sum"))
                 .sort_values("total_amount", ascending=False)
                 .head(10)
            )
top_cust

,txn_count,total_amount
CustomerID,,
CUST00394,378,"6,648,767.52"
CUST02532,424,"6,316,046.39"
CUST00611,424,"6,099,789.96"
CUST03051,369,"5,995,617.16"
CUST01335,448,"5,992,918.65"
CUST02048,453,"5,933,027.01"
CUST02769,405,"5,910,265.40"
CUST03636,352,"5,844,867.16"
CUST04653,477,"5,830,904.28"


In [38]:
print("CustAccountBalance - summary:")
print(df["CustAccountBalance"].describe().round(2))
print()
print("Negative balance rows:",(df["CustAccountBalance"]< 0).sum())
print("Zero balance rows:", (df["CustAccountBalance"] == 0).sum())

CustAccountBalance - summary:
count     580,962.00
mean       24,059.17
std        71,073.50
min             0.00
25%           716.88
50%         2,781.76
75%         8,977.33
max     1,263,280.33
Name: CustAccountBalance, dtype: float64

Negative balance rows: 0
Zero balance rows: 38


In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 580962 entries, 0 to 580961
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   TransactionID        580962 non-null  str           
 1   CustomerID           580962 non-null  str           
 2   TransactionDate      580962 non-null  datetime64[us]
 3   TransactionTime      580962 non-null  str           
 4   TransactionType      580962 non-null  str           
 5   TransactionAmount    580962 non-null  float64       
 6   TransactionCurrency  569116 non-null  str           
 7   AssetType            566487 non-null  str           
 8   CustomerDOB          580962 non-null  datetime64[us]
 9   CustGender           580962 non-null  str           
 10  CustLocation         563766 non-null  str           
 11  CustAccountBalance   580962 non-null  float64       
 12  WalletActivity       580962 non-null  int64         
 13  TransactionStatus    5809